# 11c Calibrated safety stock

In 11b the safety stock comes from the textbook formula, z times the error spread times the square root of the lead
time. That formula assumes forecast errors are unbiased, normal and independent from day to day. 07b and 11a showed
none of that holds. Here we size safety stock from the errors themselves instead, the same way 11a builds its bands:
pool each category's past 7-day error sums (scaled per product), take the quantile for the target service level, and
scale it back to the product. Then we rerun the simulation and compare.

## Load

In [1]:
import sys, warnings
sys.path.append("../src")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import ROOT, RAW, INTERIM, PROCESSED, TABLES, FIGS, FIG_STYLE
from break_detection import weekly_ca_prices
from metrics import paired_test
from pipeline import price_daily_frame, simulate_inventory, summarise_inventory, intermittency

SEED = 42
plt.rcParams.update(FIG_STYLE)
written = []
fc = pd.read_parquet(PROCESSED / "backtest_forecasts.parquet")
err = pd.read_csv(PROCESSED / "backtest_errors.csv", parse_dates=["origin"])
scales = err[["item_id", "origin", "mase_scale"]].drop_duplicates()
ORIGINS = sorted(fc["origin"].unique())
EVAL = ORIGINS[1:]
fc["category"] = fc["item_id"].str.split("_").str[0]

POL = [("lightgbm", "static"), ("lightgbm", "adaptive"), ("seasonal_naive", "static")]
f = fc[fc.set_index(["model", "variant"]).index.isin(POL)].merge(scales, on=["item_id", "origin"])
f["e"] = f["actual"] - f["forecast"]
f["r"] = f["e"] / f["mase_scale"]          # errors in units of each product's own MASE scale, so products can be pooled
print(len(ORIGINS), "origins;", f"{len(f):,} forecast days for", POL)

7 origins; 426,888 forecast days for [('lightgbm', 'static'), ('lightgbm', 'adaptive'), ('seasonal_naive', 'static')]


## The textbook version, as a check against 11b

In [2]:
long_df = pd.concat([pd.read_csv(INTERIM / f"{c}_candidates.csv", parse_dates=["date"])
                     for c in ["foods", "hobbies", "household"]], ignore_index=True)
calendar = pd.read_csv(RAW / "calendar.csv", parse_dates=["date"])
prices = pd.read_csv(RAW / "sell_prices.csv", dtype={"store_id": "category"})
price_wk = weekly_ca_prices(prices, calendar, long_df["item_id"].unique())
del prices
days = pd.date_range(EVAL[0] + pd.Timedelta(days=1), periods=28 * len(EVAL))
price_daily = price_daily_frame(price_wk, days)
inter = intermittency(long_df, ORIGINS[0])
L, SLS = 7, [0.90, 0.95, 0.99]

normal = simulate_inventory(fc, ORIGINS, price_daily, POL, SLS, L, inter)
ns = summarise_inventory(normal)
ref = pd.read_csv(TABLES / "inventory_policy_summary.csv")
chk = ns.merge(ref, on=["policy", "service_level"], suffixes=("", "_11b"))
print("largest difference from 11b: total cost", np.abs(chk["total_cost"] - chk["total_cost_11b"]).max().round(6),
      "| fill rate", np.abs(chk["fill_rate"] - chk["fill_rate_11b"]).max().round(6), "(11b rounds to 3 decimals)")

largest difference from 11b: total cost 0.000455 | fill rate 0.000389 (11b rounds to 3 decimals)


## Safety stock from the errors

In [3]:
sums = []
for (item, origin, model, variant, cat), g in f.groupby(["item_id", "origin", "model", "variant", "category"]):
    s = np.convolve(g.sort_values("date")["e"].to_numpy(), np.ones(L), mode="valid") / g["mase_scale"].iloc[0]
    sums.append(pd.DataFrame({"origin": origin, "model": model, "variant": variant, "category": cat, "r7": s}))
sums = pd.concat(sums, ignore_index=True)
scale_map = scales.set_index(["item_id", "origin"])["mase_scale"]
cat_map = long_df.groupby("item_id")["item_id"].first().str.split("_").str[0]
qcache = {}

def conformal_ss(item, model, variant, origin, lead_time, sl, past_errors):
    """Pooled, scaled 7-day error quantile from earlier origins, scaled back to the product."""
    cat = cat_map[item]
    key = (cat, model, variant, origin, sl)
    if key not in qcache:
        pool = sums[(sums["category"] == cat) & (sums["model"] == model) & (sums["variant"] == variant)
                    & (sums["origin"] < origin)]["r7"].to_numpy()
        n = len(pool)
        qcache[key] = np.quantile(pool, min(1.0, np.ceil((n + 1) * sl) / n), method="higher")
    return max(0.0, scale_map[(item, origin)] * qcache[key])

conf = simulate_inventory(fc, ORIGINS, price_daily, POL, SLS, L, inter, ss_fn=conformal_ss)
both = pd.concat([normal.assign(method="normal formula"), conf.assign(method="conformal")], ignore_index=True)
both.to_csv(TABLES / "inventory_calibrated_items.csv", index=False)
inv = summarise_inventory(both, by=("method", "policy", "service_level"))
inv.to_csv(TABLES / "inventory_calibrated_vs_normal.csv", index=False)
written += [TABLES / "inventory_calibrated_items.csv", TABLES / "inventory_calibrated_vs_normal.csv"]
inv[["method", "policy", "service_level", "fill_rate", "cycle_service_level", "cycles", "stockout_days",
     "avg_inventory_value", "holding_cost", "stockout_cost", "total_cost"]].round(4)

,method,policy,service_level,fill_rate,cycle_service_level,cycles,stockout_days,avg_inventory_value,holding_cost,stockout_cost,total_cost
0,conformal,lightgbm_adaptive,0.90,0.9793,0.8585,15293,2162,173437.5904,19957.2022,50321.1331,70278.3353
1,conformal,lightgbm_adaptive,0.95,0.9872,0.9112,15436,1234,199487.9230,22954.7747,32869.0250,55823.7997
2,conformal,lightgbm_adaptive,0.99,0.9961,0.9698,15618,334,272448.2318,31350.2075,10982.9842,42333.1917
3,conformal,lightgbm_static,0.90,0.9831,0.8653,15216,1970,170766.7714,19649.8751,40482.9408,60132.8159
4,conformal,lightgbm_static,0.95,0.9900,0.9165,15345,1135,195327.4102,22476.0308,24829.5103,47305.5411
5,conformal,lightgbm_static,0.99,0.9970,0.9733,15466,308,262175.5665,30168.1474,8574.5795,38742.7268
6,conformal,seasonal_naive_static,0.90,0.9726,0.8027,15166,3247,181120.9986,20841.3204,58673.6328,79514.9531
7,conformal,seasonal_naive_static,0.95,0.9820,0.8683,15435,1990,205453.9167,23641.2726,39204.9549,62846.2275
8,conformal,seasonal_naive_static,0.99,0.9937,0.9420,15817,649,274745.9671,31614.6044,15570.7421,47185.3465
9,normal formula,lightgbm_adaptive,0.90,0.9664,0.7711,14984,4013,150806.4975,17353.0764,74171.2334,91524.3098


In [4]:
rows = []
for (model, variant), _ in both.groupby(["model", "variant"]):
    for sl in SLS:
        p = both[(both["model"] == model) & (both["variant"] == variant) & (both["service_level"] == sl)]
        w = p.pivot(index="item_id", columns="method", values="total_cost")
        rows.append({"policy": f"{model}_{variant}", "service_level": sl,
                     **paired_test(w["conformal"] - w["normal formula"], seed=SEED)})
cmp_ = pd.DataFrame(rows)
cmp_.to_csv(TABLES / "inventory_calibrated_paired.csv", index=False)
written.append(TABLES / "inventory_calibrated_paired.csv")
print("per-product total cost, conformal minus textbook:")
cmp_[["policy", "service_level", "n", "median_diff", "ci_low", "ci_high", "share_improved", "p_value"]].round(4)

per-product total cost, conformal minus textbook:


,policy,service_level,n,median_diff,ci_low,ci_high,share_improved,p_value
0,lightgbm_adaptive,0.90,726,-12.0439,-13.6935,-10.3824,0.7769,0.0000
1,lightgbm_adaptive,0.95,726,-8.2848,-10.0326,-6.5920,0.6901,0.0000
2,lightgbm_adaptive,0.99,726,2.4520,1.5083,3.5947,0.4174,0.3902
3,lightgbm_static,0.90,726,-12.6100,-14.2554,-10.5189,0.7906,0.0000
4,lightgbm_static,0.95,726,-8.4905,-10.0923,-6.7893,0.7190,0.0000
5,lightgbm_static,0.99,726,2.5610,1.5856,3.6056,0.4187,0.1639
6,seasonal_naive_static,0.90,726,-0.7355,-1.2329,-0.2748,0.5579,0.0000
7,seasonal_naive_static,0.95,726,-1.1203,-1.9078,-0.5012,0.5909,0.0000
8,seasonal_naive_static,0.99,726,1.8271,1.3457,2.5610,0.3898,0.7372


## What we found

The textbook formula promises more than it delivers. With LightGBM forecasts and a 95% target, only 82.7% of order
cycles got through without a stockout; with a 99% target, 89.0%.

Safety stock sized from the real errors gets much closer: 91.7% of cycles at the 95% target and 97.3% at 99%. Fill
rate goes from 97.7% to 99.0% at 95%. It holds more stock ($195,327 of average inventory against $161,205), but
loses so many fewer sales that total cost falls from $68,708 to $47,306, which is 31.1% lower, and it is cheaper for
72% of products. At the 99% target the total is lower too ($38,743 against $53,492), but the typical product pays
slightly more (median +$2.56, not significant): there the saving comes from a minority of products.

These costs use our assumed prices for holding stock and losing a sale; the fill rates and cycle counts come
straight from the simulation.

## What comes next

`11d_inventory_other_seasons` checks whether the stock results hold in two other half-years.

In [5]:
# files this notebook wrote
for p in written:
    print(p.relative_to(ROOT), f"{p.stat().st_size / 1e3:,.0f} KB")

outputs\tables\inventory_calibrated_items.csv 3,220 KB
outputs\tables\inventory_calibrated_vs_normal.csv 4 KB
outputs\tables\inventory_calibrated_paired.csv 2 KB
